# Quintic Example

In this section, we'll work through the procedure for computing the roots of a
quintic in the case of a specific example with known roots, three real and two
complex, of the polynomial given by

$$
\begin{aligned}
P(x) &:= (x+3)(x+1)(x-2)(x^2+x+9) \\
&= x^5 + 3\,x^4+6\,x^3+7\,x^2-51\,x-54 \,.
\end{aligned}
$$

Since the roots are known, we can run the polynomial coefficients through our
algorithm and verify that the appropriate roots are found.

The first step is to perform a Tschirnhaus transformation, in order to get the
coefficients of the corresponding principal quintic. Taking 

$$
y = x^2+A\,x+B
$$

with $A = -6$, $B=-3$, we find 

$$
Q(y) = y^5 -11065\,y^2-88260\,y+529056\,.
$$

In [1]:
# Check of the above:
from sympy import sqrt, symbols, resultant, simplify


p1, p2, p3, p4, p5 = symbols("p_1 p_2 p_3 p_4 p_5")
x, y = symbols("x y")
P = x**5 + p1 * x**4 + p2 * x**3 + p3 * x**2 + p4 * x + p5
p_vals = {p1: 3, p2: 6, p3: 7, p4: -51, p5: -54}
B = (
    p1**2 * p2 / 2
    + 3 * p1 * p3 / 2
    + p1
    * sqrt(
        40 * p1**3 * p3
        - 15 * p1**2 * p2**2
        + 80 * p1**2 * p4
        - 190 * p1 * p2 * p3
        + 60 * p2**3
        - 200 * p2 * p4
        + 225 * p3**2
    )
    / 10
    - 2 * p2**2
) / (2 * p1**2 - 5 * p2)
B_val = B.subs(p_vals)
B_val

-3

In [2]:
A = (5 * B + p1**2 - 2 * p2) / p1
A_val = A.subs(p_vals)
A_val

-6

In [3]:
Q = resultant(P, y - x**2 - A_val * x - B_val, x).subs(p_vals)
Q

y**5 - 11065*y**2 - 88260*y + 529056

In [4]:
alpha, beta, gamma = symbols("alpha beta gamma")
q_vals = {alpha: Q.coeff(y, 2) / 5, beta: Q.coeff(y, 1) / 5, gamma: Q.coeff(y, 0)}
q_vals

{alpha: -2213, beta: -17652, gamma: 529056}

Now that the coefficients of the principal quintic have been identified, we can
proceed by solving for the values appearing in the expression

$$
y = y_r := \frac{W_r(z_0,1)f(z_0,1)}{H(z_0,1)}\left(\lambda_\pm + \mu_\pm\,\frac{t_r(z_0,1)f^2(z_0,1)}{T(z_0,1)}\right) \,.
$$

First of all, $\lambda_\pm$ is found by solving the quadratic equation

$$
\left(\alpha^4-\beta^3+\alpha\beta\gamma\right)\lambda^2 + \left(\alpha\gamma^2-2\,\beta^2\gamma-11\,\alpha^3\beta\right)\lambda + 64\,\alpha^2\beta^2-27\,\alpha^3\gamma-\beta\gamma^2 =0 \,.
$$

Solving for $\lambda$, given the values of $(\alpha,\beta,\gamma)$ above, we find

$$
\lambda = \lambda_\pm = \frac{5551876412706}{182368882091} \pm \frac{4470309903150 \sqrt{7} i}{182368882091} \,.
$$

In [5]:
# Check of the above:
l, mu = symbols("lambda mu")
l_eqn = (
    simplify(
        -(beta * l + gamma)
        * alpha
        * (
            27 * alpha**2
            + (l * alpha + 8 * beta) ** 2 / mu**2 / B
            - l * gamma
            - mu**2 * B * beta
        ).subs(mu**2, -1 / B * (l * beta + gamma) / alpha)
    )
    .expand()
    .collect(l)
)
l_eqn

-27*alpha**3*gamma + 64*alpha**2*beta**2 - beta*gamma**2 + lambda**2*(alpha**4 + alpha*beta*gamma - beta**3) + lambda*(-11*alpha**3*beta + alpha*gamma**2 - 2*beta**2*gamma)

In [6]:
from sympy import solve


l_val = solve(l_eqn.subs(q_vals), l)[0]
l_val

5551876412706/182368882091 - 4470309903150*sqrt(7)*I/182368882091

From here, the corresponding values of $I$ are

$$
\begin{aligned}
I &= I_\pm = \frac{(\alpha\lambda_\pm^2-3\,\beta\lambda_\pm-3\gamma)^3}{12^3\alpha^2(\alpha\gamma\lambda_\pm-\beta^2\lambda_\pm-\beta\gamma)} \\
&= 
\frac{15433393006320276128482185197792403047552759957120800000}{18338381126299237208601401250567585364768902047827121041} \pm \frac{2477631834917432833053465047627874055898615726894880000 \sqrt{7} i}{201722192389291609294615413756243439012457922526098331451} \,.
\end{aligned}
$$

In [7]:
# Check of the above:
I0 = (alpha * l**2 - 3 * beta * l - 3 * gamma) ** 3 / (
    1728 * alpha**2 * (alpha * gamma * l - beta**2 * l - beta * gamma)
)
I0

(alpha*lambda**2 - 3*beta*lambda - 3*gamma)**3/(1728*alpha**2*(alpha*gamma*lambda - beta**2*lambda - beta*gamma))

In [8]:
from sympy import radsimp


I_val = simplify(radsimp(I0.subs(q_vals).subs(l, l_val)))
I_val

15433393006320276128482185197792403047552759957120800000/18338381126299237208601401250567585364768902047827121041 - 2477631834917432833053465047627874055898615726894880000*sqrt(7)*I/201722192389291609294615413756243439012457922526098331451

We also need a value for $\mu$, which we take to be

$$
\begin{align*}
\mu &= \mu_\pm := \pm\sqrt{\frac{-\beta\lambda_\pm-\gamma}{\alpha B(z_0)}}\\
&=\sqrt{\frac{1106519134731851526970487630634061104470044531690575633549858636228880896}{36787850718980739066465358072503539144355288353260839478659255944041} + \frac{1965995425815591592119531912388367125043216243943615692796437685016972800 \sqrt{7} i}{36787850718980739066465358072503539144355288353260839478659255944041}} \,.
\end{align*}
$$

(Recall that $B(z_0) = 1+1/(1728\,I(z_0))$.)

In [9]:
# Check of the above:
from sympy import Rational


mu_val = (1728 * I0 * alpha**2 - 8 * alpha * l**3 - 72 * l * (beta * l + gamma)) / (
    l**2 * alpha + beta * l + gamma
)
mu_val

(-8*alpha*lambda**3 - 72*lambda*(beta*lambda + gamma) + (alpha*lambda**2 - 3*beta*lambda - 3*gamma)**3/(alpha*gamma*lambda - beta**2*lambda - beta*gamma))/(alpha*lambda**2 + beta*lambda + gamma)

In [10]:
mu_val = radsimp(mu_val.subs(q_vals).subs(l, l_val).expand())
mu_val

(-1792267617843048531497326719070295664 + 548465922790487155956308047654467600*sqrt(7)*I)/6065298897744507617042427388279571

Now, we do have to compute $z_0$ using the icosahedral equation below. (This is
the only part of the computation that can't be done using radicals.) In doing
so, we recall that the inverse is given by a ratio of hypergeometric functions:

$$
s(x) = - (1728\,x)^{1/5}\,\frac{{}_2F_1(b,1+b-c;1+b-a;1/x)}{{}_2F_1(a, 1+a-c;1+a-b;1/x)} \,.
$$

In [11]:
from sympy import hyper


def g_ratio_inf(sg_0, sg_1, sg_inf, x):
    nu_0, nu_1, nu_inf = 3, 2, 5
    a = Rational(1, 2) * (
        1
        - sg_0 * Rational(1, nu_0)
        - sg_1 * Rational(1, nu_1)
        + sg_inf * Rational(1, nu_inf)
    )
    b = Rational(1, 2) * (
        1
        - sg_0 * Rational(1, nu_0)
        - sg_1 * Rational(1, nu_1)
        - sg_inf * Rational(1, nu_inf)
    )
    c = 1 - sg_0 * Rational(1, nu_0)
    num = lambda z: z ** (-b) * hyper([b, 1 + b - c], [1 + b - a], 1 / z)
    denom = lambda z: z ** (-a) * hyper([a, 1 + a - c], [1 + a - b], 1 / z)
    return -((1728) ** Rational(1, 5)) * num(x) / denom(x)


z0_val = g_ratio_inf(1, 1, 1, I_val).evalf()
z0_val

-3.51153863228721 + 0.485762529124594*I

With $z_0 := s(I_0)$ in tow, we can now compute the roots $y_r$ of the principal
quintic:

$$
y_r = \frac{W_r(z_0,1)f(z_0,1)}{H(z_0,1)} \left(\lambda_\pm + \mu_\pm\,\frac{t_r(z_0,1)f^2(z_0,1)}{T(z_0,1)}\right) \,.
$$

In [12]:
from utils import hessian, z_simp


e, r = symbols("e r")
z_n, u, v = symbols("z_n u v")
t_0 = (
    (u**2 + v**2)
    * (u**2 - 2 * z_n * u * v - v**2)
    * (u**2 - 2 * (-1 - z_n) * u * v - v**2)
)
t_r = simplify(z_simp((e ** (15 - 3 * r) * t_0.subs(v, v * e**r)).expand(), z_n))
W_r = simplify(hessian(t_r, u, v).expand() / (-400))

In [18]:
from sympy import I, pi, exp
from icosahedron import f, T, H


def y_r(z_0, r_val):
    e_val = (exp(2 * I * pi / 5)).evalf()
    W_val = W_r.subs(u, z_0).subs(v, 1).subs(e, e_val).subs(r, r_val)
    t_val = t_r.subs(u, z_0).subs(v, 1).subs(e, e_val).subs(r, r_val)
    f_val = f.subs(u, z_0).subs(v, 1)
    H_val = H.subs(u, z_0).subs(v, 1)
    T_val = T.subs(u, z_0).subs(v, 1)
    return (W_val * f_val / H_val * (l_val + mu_val * t_val * f_val**2 / T_val)).evalf()

In [19]:
from sympy import conjugate


def ys(z_0, conj=False):
    if conj:
        return [conjugate(y_r(z_0, i)) for i in range(5)]
    return [y_r(z_0, i) for i in range(5)]


def xs(z_0, conj=False):
    yss = ys(z_0, conj=conj)
    return [solve(yss[i] - x**2 - A_val * x - B_val, x)[0] for i in range(5)]


def icosahedral_orbit(z_seed, tol=1e-10):
    """Return all 60 preimages from a regular, finite seed."""
    eps = exp(2 * I * pi / 5).evalf()
    a = eps - eps**4
    b = eps**3 - eps**2

    points = [z_seed]
    i = 0

    while i < len(points):
        z = points[i]
        candidates = (
            (eps * z).evalf(),
            ((a * z + b) / (b * z - a)).evalf(),
        )
        for candidate in candidates:
            already_seen = any(
                abs(candidate - old) <= tol * max(1, abs(candidate), abs(old))
                for old in points
            )
            if not already_seen:
                points.append(candidate)
                if len(points) > 60:
                    raise RuntimeError("Orbit did not close; try a larger tolerance.")
        i += 1

    return points

In [20]:
z_vals = icosahedral_orbit(z0_val)
len(z_vals)

60

In [21]:
for val in z_vals[:5]:
    print(ys(val))

[-8.49999999999997 + 20.7062792408486*I, -8.50000000000002 - 20.7062792408486*I, -11.0 + 5.5169898597468e-15*I, 24.0 - 2.28072769004051e-14*I, 4.0 + 5.97253095504374e-16*I]
[4.00000000000001 + 3.2816089255705e-15*I, -8.49999999999997 + 20.7062792408486*I, -8.50000000000002 - 20.7062792408486*I, -11.0 + 2.99115116356163e-15*I, 24.0 - 2.01198649754653e-14*I]
[-8.49999999999997 + 20.7062792408486*I, -11.0 + 1.62564325065576e-14*I, -8.5 - 20.7062792408486*I, 3.99999999999999 - 1.54896305200672e-14*I, 24.0 - 1.91632733986813e-16*I]
[24.0 - 2.95001653313753e-14*I, 4.00000000000001 - 2.21564845463706e-15*I, -8.49999999999998 + 20.7062792408487*I, -8.50000000000001 - 20.7062792408486*I, -11.0 + 2.10476167744611e-15*I]
[4.00000000000001 - 9.60273798412264e-15*I, -8.50000000000001 - 20.7062792408486*I, -8.49999999999997 + 20.7062792408486*I, 24.0 - 2.03325854417069e-14*I, -11.0 + 2.07939134660995e-15*I]


In [28]:
from sympy import Abs, N


def root_permutation(reference, roots, tol=1e-8):
    """Return labels p such that roots[i] ≈ reference[p[i] - 1]."""
    if len(reference) != len(roots):
        raise ValueError("Root lists must have equal lengths.")

    permutation = []
    for root in roots:
        matches = [
            j + 1 for j, ref in enumerate(reference) if Abs(N(root - ref, 25)) < tol
        ]
        if len(matches) != 1:
            raise ValueError("Root match is missing or ambiguous.")
        permutation.append(matches[0])

    if len(set(permutation)) != len(reference):
        raise ValueError("Root matching is not bijective.")

    return tuple(permutation)

In [29]:
all_permutations = []
for val in z_vals:
    all_permutations.append(root_permutation(ys(val), ys(z_vals[0])))
    all_permutations.append(root_permutation(ys(val, conj=True), ys(z_vals[0])))

In [24]:
all_permutations = []
for val in z_vals:
    all_permutations.append(root_permutation(xs(val), xs(z_vals[0])))
    all_permutations.append(root_permutation(xs(val, conj=True), xs(z_vals[0])))

In [30]:
from itertools import permutations

exp_perm = []
for x in permutations(range(1, 6)):
    exp_perm.append(x)

len(exp_perm)

120

In [31]:
exp_perm[:6]

[(1, 2, 3, 4, 5),
 (1, 2, 3, 5, 4),
 (1, 2, 4, 3, 5),
 (1, 2, 4, 5, 3),
 (1, 2, 5, 3, 4),
 (1, 2, 5, 4, 3)]

In [32]:
for perm in all_permutations:
    if perm in exp_perm:
        exp_perm.remove(perm)
    else:
        print("fail")

len(exp_perm)

0

In [ ]:
y_vals = [y_r(z0_val, i) for i in range(5)]
y_vals

[-8.49999999999997 + 20.7062792408486*I,
 -8.50000000000002 - 20.7062792408486*I,
 -11.0 + 5.5169898597468e-15*I,
 24.0 - 2.28072769004051e-14*I,
 4.0 + 5.97253095504374e-16*I]

In [ ]:
x_vals = [solve(y_vals[i] - x**2 - A_val * x - B_val, x) for i in range(5)]
x_vals

[[-0.499999999999996 - 2.9580398915498*I, 6.5 + 2.9580398915498*I],
 [-0.499999999999996 + 2.9580398915498*I, 6.5 - 2.9580398915498*I],
 [2.0 - 2.7584949298734e-15*I, 4.0 + 2.7584949298734e-15*I],
 [-3.0 + 1.90060640836709e-15*I, 9.0 - 1.90060640836709e-15*I],
 [-1.0 - 7.46566369380468e-17*I, 7.0 + 7.46566369380468e-17*I]]

In [ ]:
from utils import reduce_multiply

reduce_multiply([(x - x_vals[i][1]) for i in range(5)]).expand()

x**5 - 33.0*x**4 - 9.32545158444355e-16*I*x**4 + 438.0*x**3 + 3.63228717259076e-14*I*x**3 - 2923.0*x**2 - 4.8541284381788e-13*I*x**2 + 9752.99999999999*x + 2.83651493898038e-12*I*x - 12852.0 - 6.28604784395327e-12*I

In [ ]:
reduce_multiply([(x - x_vals[i][0]) for i in range(5)]).expand()

x**5 + 2.99999999999999*x**4 + 8.38585487671331e-16*I*x**4 + 5.99999999999995*x**3 + 1.3471989569378e-14*I*x**3 + 6.99999999999997*x**2 + 3.19716414530058e-14*I*x**2 - 50.9999999999998*x + 1.26823872347167e-13*I*x - 53.9999999999998 + 1.03454176581213e-13*I

In [ ]:
reduce_multiply([(y - y_vals[i]) for i in range(2)]).expand()

y**2 + 17.0*y - 2.1316282072803e-14*I*y + 500.999999999999 - 1.22213350550737e-12*I

In [ ]:
reduce_multiply([(x - x_vals[i][1]) for i in range(2)]).expand()

x**2 - 13.0*x + 50.9999999999999

In [ ]:
((x - x_vals[0][0]) * (x - x_vals[-1][0])).expand()

x**2 + 0.999999999999992*x + 8.99999999999997

In [ ]:
from utils import e_simp, hessian

u, v, e = symbols("u v e")
z_n = e**2 + e**3
z_m = e + e**4


def tI(u, v):
    return (
        (u**2 + v**2)
        * (u**2 - 2 * z_n * u * v - v**2)
        * (u**2 - 2 * z_m * u * v - v**2)
    )


def t_r(r, z):
    return e_simp(e ** (15 - 3 * r) * tI(z, v * e**r), e)


def W_r(r, z):
    return (
        simplify(e_simp(hessian(t_r(r, u), u, v).expand() / (-400), e))
        .expand()
        .subs(u, z)
        .subs(v, 1)
    )


t_r(2, u).subs(u, z0_val).evalf()

e**3*(0.790658060990224 + 0.109374294176339*I)*(2.0*v**5 + v**2*(-2.32948406352381 - 1.0190718926995*I) - 0.250426627682757 - 0.205553716661*I) + e**2*(0.790658060990224 + 0.109374294176339*I)*(2.0*v**5 + v**4*(-3.95329030495112 - 0.546871470881693*I) - 0.250426627682757 - 0.205553716661*I) + e*(v**6 + v**5*(1.58131612198045 + 0.218748588352677*I) + v*(0.500853255365515 + 0.411107433322001*I) - 0.17551953918885 - 0.189912938690283*I) + v**5*(1.58131612198045 + 0.218748588352677*I) - 0.17551953918885 - 0.189912938690283*I

In [ ]:
W_r(2, u).subs(u, z0_val).evalf()

e**3 + e**3*(-4.29224203227574 - 1.21068734297894*I) + e**3*(0.790658060990224 + 0.109374294176339*I)**7 + e**3*(1.7529863937793 + 1.438876016627*I) + e**2*(-3.26127768893333 - 1.4267006497793*I) + e**2*(-4.29224203227574 - 1.21068734297894*I) + e**2*(0.790658060990224 + 0.109374294176339*I)**8 + e**2*(0.790658060990224 + 0.109374294176339*I)**7 + e*(-4.29224203227574 - 1.21068734297894*I) + e*(0.790658060990224 + 0.109374294176339*I) + e*(0.790658060990224 + 0.109374294176339*I)**7 + e*(1.22863677432195 + 1.32939057083198*I) - 4.29224203227574 - 1.21068734297894*I + (0.790658060990224 + 0.109374294176339*I)**7

In [ ]:
from icosahedron import H, f, T

u, v = symbols("u v")
T.subs(u, z0_val).subs(v, 1).expand()

-234.364217146032 - 1180.03038727335*I

In [ ]:
# Check of the above:
from sympy import Add, Eq, Poly, discriminant, resultant, solve, symbols

x, p1, p2, p3, p4, p5 = symbols("x p1 p2 p3 p4 p5")
expr = x**5 + p1 * x**4 + p2 * x**3 + p3 * x**2 + p4 * x + p5
Eq(expr, 0)

Eq(p1*x**4 + p2*x**3 + p3*x**2 + p4*x + p5 + x**5, 0)

In [ ]:
y, A, B, C = symbols("y A B C")

expr2 = Poly(resultant(expr, y - x**2 - A * x - C, x), y).terms()
expr2 = Add(*[b * y ** a[0] for a, b in expr2], evaluate=False)
expr2

y**5 + y**4*(A*p1 - 5*C - p1**2 + 2*p2) + y**3*(A**2*p2 - 4*A*C*p1 - A*p1*p2 + 3*A*p3 + 10*C**2 + 4*C*p1**2 - 8*C*p2 - 2*p1*p3 + p2**2 + 2*p4) + y**2*(A**3*p3 - 3*A**2*C*p2 - A**2*p1*p3 + 4*A**2*p4 + 6*A*C**2*p1 + 3*A*C*p1*p2 - 9*A*C*p3 - 3*A*p1*p4 + A*p2*p3 + 5*A*p5 - 10*C**3 - 6*C**2*p1**2 + 12*C**2*p2 + 6*C*p1*p3 - 3*C*p2**2 - 6*C*p4 - 2*p1*p5 + 2*p2*p4 - p3**2) + y*(A**4*p4 - 2*A**3*C*p3 - A**3*p1*p4 + 5*A**3*p5 + 3*A**2*C**2*p2 + 2*A**2*C*p1*p3 - 8*A**2*C*p4 - 4*A**2*p1*p5 + A**2*p2*p4 - 4*A*C**3*p1 - 3*A*C**2*p1*p2 + 9*A*C**2*p3 + 6*A*C*p1*p4 - 2*A*C*p2*p3 - 10*A*C*p5 + 3*A*p2*p5 - A*p3*p4 + 5*C**4 + 4*C**3*p1**2 - 8*C**3*p2 - 6*C**2*p1*p3 + 3*C**2*p2**2 + 6*C**2*p4 + 4*C*p1*p5 - 4*C*p2*p4 + 2*C*p3**2 - 2*p3*p5 + p4**2) + (A**5*p5 - A**4*C*p4 - A**4*p1*p5 + A**3*C**2*p3 + A**3*C*p1*p4 - 5*A**3*C*p5 + A**3*p2*p5 - A**2*C**3*p2 - A**2*C**2*p1*p3 + 4*A**2*C**2*p4 + 4*A**2*C*p1*p5 - A**2*C*p2*p4 - A**2*p3*p5 + A*C**4*p1 + A*C**3*p1*p2 - 3*A*C**3*p3 - 3*A*C**2*p1*p4 + A*C**2*p2*p3 + 5

In [ ]:
from sympy import Rational

expr3 = expr2.subs(p1, 0).subs(p3, 0).subs(C, 2 * p2 * Rational(1, 5))
expr3

A**5*p5 - 2*A**4*p2*p4/5 - A**3*p2*p5 - 8*A**2*p2**4/125 + 6*A**2*p2**2*p4/25 - 2*A*p2**2*p5/5 + A*p4*p5 - 72*p2**5/3125 + 24*p2**3*p4/125 - 2*p2*p4**2/5 - p5**2 + y**5 + y**3*(A**2*p2 - 3*p2**2/5 + 2*p4) + y**2*(-6*A**2*p2**2/5 + 4*A**2*p4 + 5*A*p5 + 2*p2**3/25 - 2*p2*p4/5) + y*(A**4*p4 + 5*A**3*p5 + 12*A**2*p2**3/25 - 11*A**2*p2*p4/5 - A*p2*p5 + 12*p2**4/125 - 16*p2**2*p4/25 + p4**2)

In [ ]:
A2val = solve(expr3.coeff(y**3), A**2)[0]
A2val

3*p2/5 - 2*p4/p2

In [ ]:
q4, q5 = symbols("q_4 q_5")
A2val.subs(p2, 10 * B).subs(p4, 45 * B**2 - 5 * B * q4).expand()

-3*B + q_4

In [ ]:
expr4 = (
    expr3
    # .subs(A**5, A2val**2 * A)
    # .subs(A**4, A2val**2)
    # .subs(A**3, A2val * A)
    # .subs(A**2, A2val)
    .subs(A**2 * p2, (A2val * p2).expand())
    # .subs(A**2*p2**2, (A2val*p2**2).expand())
    # .subs(p4,Rational(45,100)*p2**2+q4)
    # .subs(p5,p2**2*Rational(1,100)+q5)
    # .subs(p2,10*B)
    .expand()
    .collect(y)
)
expr4

A**5*p5 - 6*A**2*p2**2*p4/25 + 4*A**2*p4**2/5 - A*p2**2*p5 + 3*A*p4*p5 - 192*p2**5/3125 + 58*p2**3*p4/125 - 22*p2*p4**2/25 - p5**2 + y**5 + y**2*(4*A**2*p4 + 5*A*p5 - 16*p2**3/25 + 2*p2*p4) + y*(A**4*p4 + 5*A**3*p5 - A*p2*p5 + 48*p2**4/125 - 73*p2**2*p4/25 + 27*p4**2/5)

In [ ]:
alpha = Rational(1, 5) * expr4.coeff(y, 2)
alpha

4*A**2*p4/5 + A*p5 - 16*p2**3/125 + 2*p2*p4/5

In [ ]:
alpha.subs(A**2, A2val).subs(p2, 10 * B).subs(p4, 45 * B**2).subs(p5, B**2)

A*B**2 - 56*B**3

In [ ]:
beta = Rational(1, 5) * expr4.coeff(y, 1)
beta

A**4*p4/5 + A**3*p5 - A*p2*p5/5 + 48*p2**4/625 - 73*p2**2*p4/125 + 27*p4**2/25

In [ ]:
beta.subs(A**2, A2val).subs(A**4, A2val**2).subs(A**3, A2val * A).subs(p2, 10 * B).subs(
    p4, 45 * B**2
).subs(p5, B**2)

-5*A*B**3 + 408*B**4

In [ ]:
gamma = expr4.coeff(y, 0)
gamma

A**5*p5 - 6*A**2*p2**2*p4/25 + 4*A**2*p4**2/5 - A*p2**2*p5 + 3*A*p4*p5 - 192*p2**5/3125 + 58*p2**3*p4/125 - 22*p2*p4**2/25 - p5**2

In [ ]:
gamma.subs(A**5, A2val**2 * A).subs(A**2, A2val).subs(A**4, A2val**2).subs(
    A**3, A2val * A
).subs(p2, 10 * B).subs(p4, 45 * B**2).subs(p5, B**2)

44*A*B**4 - 4704*B**5 - B**4

In [ ]:
gamma2 = (
    gamma.subs(A**5, (-3 * B + q4) ** 2 * A)
    .subs(A**2, -3 * B + q4)
    .subs(p5, B**2 + q5)
    .subs(p4, 45 * B**2 - 5 * B * q4)
    .subs(p2, 10 * B)
    .expand()
)
gamma2

44*A*B**4 - 21*A*B**3*q_4 + A*B**2*q_4**2 + 44*A*B**2*q_5 - 21*A*B*q_4*q_5 + A*q_4**2*q_5 - 4704*B**5 + 2900*B**4*q_4 - B**4 - 520*B**3*q_4**2 + 20*B**2*q_4**3 - 2*B**2*q_5 - q_5**2

In [ ]:
gamma_B = gamma2.subs(q4, 0).subs(q5, 0)
gamma_B

44*A*B**4 - 4704*B**5 - B**4

In [ ]:
beta2 = (
    beta.subs(A**4, (-3 * B + q4) ** 2)
    .subs(A**3, (-3 * B + q4) * A)
    .subs(p4, 45 * B**2 - 5 * B * q4)
    .subs(p5, B**2 + q5)
    .subs(p2, 10 * B)
    .expand()
)
beta2

-5*A*B**3 + A*B**2*q_4 - 5*A*B*q_5 + A*q_4*q_5 + 408*B**4 - 257*B**3*q_4 + 42*B**2*q_4**2 - B*q_4**3

In [ ]:
beta_B = beta2.subs(q4, 0).subs(q5, 0)
beta_B

-5*A*B**3 + 408*B**4

In [ ]:
alpha2 = (
    alpha.subs(A**2, -3 * B + q4)
    .subs(p4, 45 * B**2 - 5 * B * q4)
    .subs(p2, 10 * B)
    .subs(p5, B**2 + q5)
    .expand()
)
alpha2

A*B**2 + A*q_5 - 56*B**3 + 28*B**2*q_4 - 4*B*q_4**2

In [ ]:
alpha_B = alpha2.subs(q4, 0).subs(q5, 0)
alpha_B

A*B**2 - 56*B**3

In [ ]:
l = symbols("lambda")
alpha, beta, gamma = symbols("alpha beta gamma")
l_eqn = (
    (
        (alpha**4 - beta**3 + alpha * beta * gamma) * l**2
        + (alpha * gamma**2 - 2 * beta**2 * gamma - 11 * alpha**3 * beta) * l
        + 64 * alpha**2 * beta**2
        - 27 * alpha**3 * gamma
        - beta * gamma**2
    )
    .expand()
    # .subs(A**2, A2val)
    # .subs(A**4, A2val**2)
    # .subs(A**3, A2val * A)
    # .expand()
    .collect(l)
)
l_eqn

-27*alpha**3*gamma + 64*alpha**2*beta**2 - beta*gamma**2 + lambda**2*(alpha**4 + alpha*beta*gamma - beta**3) + lambda*(-11*alpha**3*beta + alpha*gamma**2 - 2*beta**2*gamma)

In [ ]:
l_eqn_B = (
    l_eqn.subs(alpha, alpha_B)
    .subs(beta, beta_B)
    .subs(gamma, gamma_B)
    .expand()
    .collect(A)
)
l_eqn_B

A**4*(412*B**10 + 55*B**9*lambda + B**8*lambda**2) + A**3*(-104048*B**11 - 13992*B**10*lambda + 27*B**10 - 319*B**9*lambda**2) + A**2*(9543040*B**12 + 1343296*B**11*lambda - 4976*B**11 + 42008*B**10*lambda**2 - 38*B**10*lambda + 5*B**9*lambda**2) + A*(-329032704*B**13 - 59606528*B**12*lambda + 336960*B**12 - 2447168*B**11*lambda**2 + 6176*B**11*lambda + 5*B**11 - 688*B**10*lambda**2 + B**10*lambda) + 2077286400*B**14 + 1115111424*B**13*lambda - 8580096*B**13 + 49394176*B**12*lambda**2 - 193920*B**12*lambda - 408*B**12 + 22848*B**11*lambda**2 - 56*B**11*lambda

In [ ]:
def A_simp(expr, A, B):
    poly = Poly(expr, A)
    coeffs = poly.all_coeffs()
    while len(coeffs) > 2:
        top_factor = coeffs[0]
        coeffs[2] += (-3 * B) * top_factor
        coeffs.pop(0)
    return Poly.from_list(coeffs, A).as_expr()


l_eqn_B2 = A_simp(l_eqn_B, A, B).expand().collect(l)
l_eqn_B2

-329032704*A*B**13 + 649104*A*B**12 - 76*A*B**11 + 2077286400*B**14 - 37209216*B**13 + 18228*B**12 + lambda**2*(-2447168*A*B**11 + 269*A*B**10 + 49394176*B**12 - 103176*B**11 - 6*B**10) + lambda*(-59606528*A*B**12 + 48152*A*B**11 + A*B**10 + 1115111424*B**13 - 4223808*B**12 + 553*B**11)

In [ ]:
discriminant(l_eqn_B, l).factor()

B**18*(A - 56*B)**2*(1377*A**6 - 442992*A**5*B - 108*A**5 + 49438656*A**4*B**2 + 29840*A**4*B - 2211784704*A**3*B**3 - 3595776*A**3*B**2 - 450*A**3*B + 60791648256*A**2*B**4 + 257292288*A**2*B**3 + 98736*A**2*B**2 - 5689087819776*A*B**5 - 12616777728*A*B**4 - 8007552*A*B**3 - 176*A*B**2 + 265640971862016*B**6 + 342122102784*B**5 + 247919616*B**4 + 18816*B**3 + B**2)

In [ ]:
A_simp(discriminant(l_eqn_B2, l), A, B).factor()

-B**21*(1728*B + 1)**2*(15938721792*A*B**2 - 11889568*A*B + 718*A - 278986788864*B**3 + 1062185088*B**2 - 219409*B + 3)

In [ ]:
l_vals = solve(l_eqn_B2, l)
l_vals[0].factor()

-(59606528*A*B**2 - 48152*A*B - A - 1115111424*B**3 + 4223808*B**2 - 553*B + sqrt(332144963485696*A**2*B**4 + 967558266880*A**2*B**3 + 757027072*A**2*B**2 + 178080*A**2*B + A**2 - 47592768251363328*A*B**5 - 19582162698240*A*B**4 + 23007688704*A*B**3 + 9408160*A*B**2 - 718*A*B + 833050087759282176*B**6 - 1211054445035520*B**5 + 165901688832*B**4 + 1958215680*B**3 + 743281*B**2))/(2*(2447168*A*B - 269*A - 49394176*B**2 + 103176*B + 6))

In [ ]:
l_vals[1]

(-59606528*A*B**2 + 48152*A*B + A + 1115111424*B**3 - 4223808*B**2 + 553*B + sqrt(332144963485696*A**2*B**4 + 967558266880*A**2*B**3 + 757027072*A**2*B**2 + 178080*A**2*B + A**2 - 47592768251363328*A*B**5 - 19582162698240*A*B**4 + 23007688704*A*B**3 + 9408160*A*B**2 - 718*A*B + 833050087759282176*B**6 - 1211054445035520*B**5 + 165901688832*B**4 + 1958215680*B**3 + 743281*B**2))/(2*(2447168*A*B - 269*A - 49394176*B**2 + 103176*B + 6))